# Security SLM Fine-Tune
## DeepSeek-R1-Distill-Qwen-1.5B → Blue/Red Team Security Model

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Left sidebar → Secrets (key icon) → add `HUGGINGFACE_TOKEN` and `WANDB_API_KEY`
3. Upload `security_dataset.jsonl` to Colab file browser (left sidebar → folder icon)

**Run cells top to bottom. Do NOT skip the baseline test cells.**

| Phase | Cells | Purpose |
|-------|-------|---------|
| Setup | 1-3 | Install, auth, GPU check |
| Baseline | 4-5 | Load base model, run pre-training eval |
| Train | 6-8 | LoRA, dataset, train |
| Compare | 9 | Run same eval on fine-tuned model, show diff |
| Export | 10-13 | Push to HF, export GGUF |

In [ ]:
# ── Cell 1: Install ──────────────────────────────────────────
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install wandb huggingface_hub
print('Install complete.')

In [ ]:
# ── Cell 2: Authenticate WandB + HuggingFace ─────────────────
import wandb
from huggingface_hub import login
from google.colab import userdata

wandb.login(key=userdata.get('WANDB_API_KEY'))
wandb.init(
    project='security-slm',
    name='deepseek-r1-1.5b-finetune',
    config={
        'base_model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
        'lora_rank': 16,
        'dataset_size': 20,
        'quantization': 'q4_k_m',
    }
)

login(token=userdata.get('HUGGINGFACE_TOKEN'))

# CHANGE THIS to your HuggingFace username/repo
HF_REPO = 'your-username/security-slm-1.5b'

print(f'Authenticated. Target repo: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── Cell 3: Check GPU ─────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## PHASE 1 — Baseline Test (Before Fine-Tuning)

Run the **unmodified** base model against 5 security prompts.
Results are saved to `baseline_results.json` and logged to WandB.
We re-run the same prompts after training to measure improvement.

In [ ]:
# ── Cell 4: Load base model ───────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('Base model loaded.')

In [ ]:
# ── Cell 5: BASELINE EVALUATION ──────────────────────────────
import json, re, textwrap

# 5 prompts covering the core security categories
EVAL_PROMPTS = [
    {
        'id': 'mcp_red',
        'category': 'MCP Security',
        'prompt': 'Explain how an attacker can craft a malicious MCP server tool description to trick an AI agent into exfiltrating environment variables to an external server.'
    },
    {
        'id': 'prompt_hijack',
        'category': 'Prompt Hijacking',
        'prompt': 'What is the Payload Splitting technique and how does it bypass LlamaGuard by splitting a harmful instruction across multiple conversation turns?'
    },
    {
        'id': 'agentic_blue',
        'category': 'Agentic Security',
        'prompt': 'Implement a Python circuit breaker for an AI agent that stops execution when the same tool is called more than 3 times consecutively, preventing resource exhaustion attacks.'
    },
    {
        'id': 'ssrf_red',
        'category': 'Cloud-AI SSRF',
        'prompt': 'How can an attacker exploit an LLM with a URL-fetching tool to steal temporary AWS credentials from the EC2 instance metadata endpoint at 169.254.169.254?'
    },
    {
        'id': 'iam_blue',
        'category': 'Cloud IAM',
        'prompt': 'Write a minimal AWS IAM policy in JSON that gives an AI agent read-only access to s3://company-data/ai-agent/ only, with an explicit deny for all destructive actions.'
    },
]

def score_response(response: str) -> dict:
    """Score a model response on 5 dimensions."""
    has_think = '<think>' in response
    think_steps = 0
    if has_think:
        think_block = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
        if think_block:
            think_steps = len([l for l in think_block.group(1).split('\n') if l.strip()])

    # Technical depth: presence of concrete technical markers
    tech_markers = [
        r'\b(AWS|IAM|S3|EC2|arn:|s3://)',
        r'\b(def |class |import |```python)',
        r'\b(CVE|OWASP|RFC|NIST)',
        r'\b(json\.dumps|requests\.|boto3|aws_|iam:)',
        r'\b(169\.254|metadata|SSRF|injection|exfiltrat)',
    ]
    tech_score = sum(1 for p in tech_markers if re.search(p, response))

    response_len = len(response.split())

    # Composite score /10
    score = 0
    score += 3 if has_think else 0           # Has reasoning block
    score += min(3, think_steps // 2)        # Quality of reasoning (max 3)
    score += min(2, tech_score)              # Technical depth (max 2)
    score += 2 if response_len > 150 else 1 if response_len > 50 else 0  # Length

    return {
        'has_think_block': has_think,
        'think_steps': think_steps,
        'tech_depth_score': tech_score,
        'word_count': response_len,
        'total_score': score,
    }

def run_inference(model, tokenizer, prompt: str, max_new_tokens: int = 512) -> str:
    FastLanguageModel.for_inference(model)
    formatted = f'### Instruction:\n{prompt}\n\n### Response:\n'
    inputs = tokenizer(formatted, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Return only the response part
    return response.split('### Response:\n', 1)[-1].strip()

# ── Run baseline ──
print('Running BASELINE evaluation on unmodified base model...')
print('=' * 60)
baseline_results = []

for item in EVAL_PROMPTS:
    print(f"\n[{item['category']}] {item['prompt'][:80]}...")
    response = run_inference(model, tokenizer, item['prompt'])
    scores = score_response(response)
    result = {
        'id': item['id'],
        'category': item['category'],
        'prompt': item['prompt'],
        'response': response,
        'scores': scores,
        'model': 'baseline',
    }
    baseline_results.append(result)

    print(f'  Score: {scores["total_score"]}/10 | '
          f'<think>: {scores["has_think_block"]} | '
          f'Steps: {scores["think_steps"]} | '
          f'Words: {scores["word_count"]}')
    print(f'  Response preview: {response[:200]}...')

# Save baseline to file
with open('baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

baseline_avg = sum(r['scores']['total_score'] for r in baseline_results) / len(baseline_results)
baseline_think_rate = sum(1 for r in baseline_results if r['scores']['has_think_block']) / len(baseline_results)

print('\n' + '=' * 60)
print(f'BASELINE SUMMARY')
print(f'  Average score:    {baseline_avg:.1f}/10')
print(f'  <think> rate:     {baseline_think_rate*100:.0f}%')
print(f'  Results saved to: baseline_results.json')

# Log to WandB
wandb.log({
    'baseline/avg_score': baseline_avg,
    'baseline/think_block_rate': baseline_think_rate,
    **{f'baseline/score_{r["id"]}': r['scores']['total_score'] for r in baseline_results}
})

---
## PHASE 2 — Fine-Tuning

In [ ]:
# ── Cell 6: Apply LoRA ────────────────────────────────────────
# Switch model back to training mode after inference
FastLanguageModel.for_training(model)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA applied.')
model.print_trainable_parameters()

In [ ]:
# ── Cell 7: Load + format dataset ────────────────────────────
from datasets import load_dataset

def format_sample(sample):
    return {
        'text': f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['content']}"
    }

dataset = load_dataset('json', data_files='security_dataset.jsonl', split='train')
dataset = dataset.map(format_sample)
print(f'Dataset: {len(dataset)} samples')

In [ ]:
# ── Cell 8: Train ─────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        save_steps=25,
        save_total_limit=2,
        output_dir='outputs',
        optim='adamw_8bit',
        seed=42,
        report_to='wandb',
        run_name='security-slm-run-1',
    ),
)

print('Training...')
trainer_stats = trainer.train()
wandb.finish()
print(f'Done. Final loss: {trainer_stats.training_loss:.4f}')

---
## PHASE 3 — Post-Training Evaluation & Comparison

Run the **same 5 prompts** on the fine-tuned model and compare against baseline.

In [ ]:
# ── Cell 9: POST-TRAINING EVALUATION + COMPARISON ────────────
import json, re

# Load baseline results
with open('baseline_results.json') as f:
    baseline_results = json.load(f)
baseline_by_id = {r['id']: r for r in baseline_results}

print('Running POST-TRAINING evaluation...')
print('=' * 60)
finetuned_results = []

for item in EVAL_PROMPTS:
    print(f"\n[{item['category']}]")
    response = run_inference(model, tokenizer, item['prompt'])
    scores = score_response(response)
    result = {
        'id': item['id'],
        'category': item['category'],
        'prompt': item['prompt'],
        'response': response,
        'scores': scores,
        'model': 'finetuned',
    }
    finetuned_results.append(result)

with open('finetuned_results.json', 'w') as f:
    json.dump(finetuned_results, f, indent=2)

# ── Side-by-side comparison table ──
print('\n' + '=' * 70)
print(f'{"COMPARISON: Baseline vs Fine-Tuned":^70}')
print('=' * 70)
print(f'{"Category":<22} {"Base Score":>10} {"FT Score":>8} {"Delta":>6} {"Think%":>7}')
print('-' * 70)

total_base, total_ft = 0, 0
for ft in finetuned_results:
    base = baseline_by_id[ft['id']]
    base_s = base['scores']['total_score']
    ft_s = ft['scores']['total_score']
    delta = ft_s - base_s
    delta_str = f'+{delta}' if delta >= 0 else str(delta)
    base_think = 'YES' if base['scores']['has_think_block'] else 'NO'
    ft_think = 'YES' if ft['scores']['has_think_block'] else 'NO'
    print(f"{ft['category']:<22} {base_s:>4}/10     {ft_s:>4}/10  {delta_str:>5}   {base_think}->{ft_think}")
    total_base += base_s
    total_ft += ft_s

n = len(finetuned_results)
avg_base = total_base / n
avg_ft = total_ft / n
improvement = ((avg_ft - avg_base) / max(avg_base, 0.01)) * 100

print('=' * 70)
print(f'{"AVERAGE":<22} {avg_base:>4.1f}/10     {avg_ft:>4.1f}/10  {"+" if improvement >= 0 else ""}{improvement:.0f}%')
print('=' * 70)

think_base = sum(1 for r in baseline_results if r['scores']['has_think_block']) / n * 100
think_ft = sum(1 for r in finetuned_results if r['scores']['has_think_block']) / n * 100
print(f'\n<think> block rate:  Baseline {think_base:.0f}%  ->  Fine-tuned {think_ft:.0f}%')

avg_words_base = sum(r['scores']['word_count'] for r in baseline_results) / n
avg_words_ft = sum(r['scores']['word_count'] for r in finetuned_results) / n
print(f'Avg response length: Baseline {avg_words_base:.0f} words  ->  Fine-tuned {avg_words_ft:.0f} words')

# Log comparison to WandB
wandb.init(project='security-slm', name='eval-comparison', resume='allow')
wandb.log({
    'eval/baseline_avg_score': avg_base,
    'eval/finetuned_avg_score': avg_ft,
    'eval/score_improvement_pct': improvement,
    'eval/baseline_think_rate': think_base,
    'eval/finetuned_think_rate': think_ft,
    **{f'eval/ft_score_{r["id"]}': r['scores']['total_score'] for r in finetuned_results}
})
wandb.finish()

# ── Print one full response comparison (first prompt) ──
print('\n' + '=' * 70)
print('FULL RESPONSE COMPARISON — MCP Security prompt')
print('=' * 70)
print('--- BASELINE ---')
print(baseline_by_id['mcp_red']['response'][:800])
print('\n--- FINE-TUNED ---')
print(next(r for r in finetuned_results if r['id'] == 'mcp_red')['response'][:800])

---
## PHASE 4 — Export & Push

In [ ]:
# ── Cell 10: Push LoRA adapter to HuggingFace (crash-safe) ───
model.save_pretrained('security-slm-lora')
tokenizer.save_pretrained('security-slm-lora')
model.push_to_hub(HF_REPO, token=userdata.get('HUGGINGFACE_TOKEN'))
tokenizer.push_to_hub(HF_REPO, token=userdata.get('HUGGINGFACE_TOKEN'))
print(f'Adapter saved: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── Cell 11: Export to GGUF Q4_K_M ───────────────────────────
print('Exporting GGUF Q4_K_M (~5 min)...')
model.save_pretrained_gguf('security-slm', tokenizer, quantization_method='q4_k_m')

import os
size_gb = os.path.getsize('security-slm-unsloth.Q4_K_M.gguf') / 1e9
print(f'Done: security-slm-unsloth.Q4_K_M.gguf ({size_gb:.2f} GB)')

In [ ]:
# ── Cell 12: Push GGUF to HuggingFace ────────────────────────
from huggingface_hub import HfApi
api = HfApi()
print('Uploading GGUF (~1.2GB)...')
api.upload_file(
    path_or_fileobj='security-slm-unsloth.Q4_K_M.gguf',
    path_in_repo='security-slm-unsloth.Q4_K_M.gguf',
    repo_id=HF_REPO,
    token=userdata.get('HUGGINGFACE_TOKEN'),
)
# Also upload eval results
for fname in ['baseline_results.json', 'finetuned_results.json']:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=f'eval/{fname}',
        repo_id=HF_REPO,
        token=userdata.get('HUGGINGFACE_TOKEN'),
    )
print(f'All files uploaded: https://huggingface.co/{HF_REPO}')

In [ ]:
# ── Cell 13: (Fallback) Direct download if HF upload fails ───
# Only run this cell if Cell 12 failed
from google.colab import files
files.download('security-slm-unsloth.Q4_K_M.gguf')
files.download('baseline_results.json')
files.download('finetuned_results.json')

---
## Local Inference — Run on your 4GB RAM machine

### Deploy with Ollama
```bash
# Pull from HuggingFace (after Cell 12 completes)
ollama pull hf.co/your-username/security-slm-1.5b
ollama run your-username/security-slm-1.5b
```

### Or from local GGUF file
```bash
cat > Modelfile << 'EOF'
FROM ./security-slm-unsloth.Q4_K_M.gguf
SYSTEM "You are an elite AI security researcher specializing in offensive and defensive security for AI systems, cloud-native environments, and agentic AI architectures."
PARAMETER num_ctx 2048
PARAMETER temperature 0.3
EOF

ollama create security-slm -f Modelfile
ollama run security-slm
```

### Run local comparison script
```bash
# After downloading baseline_results.json and the GGUF:
python eval/compare_local.py
```

---
### Score Interpretation
| Metric | Baseline expectation | Fine-tuned target |
|--------|---------------------|-------------------|
| Avg score | 2-5/10 | 7-9/10 |
| `<think>` rate | 20-60% | 90-100% |
| Avg word count | 50-150 | 200-500 |
| Tech depth | 1-2/5 | 4-5/5 |

### Training Loss Interpretation
| Final Loss | Meaning | Action |
|-----------|---------|--------|
| > 1.5 | Underfit | Increase epochs or r=32 |
| 0.5-1.5 | Good | Proceed |
| < 0.3 | Possible overfit | Reduce epochs next run |